# Session 7: Diagnostic Accuracy

**Module 3: Programming for Biological Data**  
**Date:** January 26, 2026 | 18:30 – 21:30  
**Instructor:** Dr. Haogao Gu

---

## Learning Objectives

By the end of this session, you will be able to:
1. Understand the **confusion matrix** and its components
2. Calculate **sensitivity, specificity, PPV, and NPV**
3. Generate and interpret **ROC curves**
4. Use the **pROC** package to find optimal cutoffs

In [ ]:
# ============================================
# STANDARD SETUP PROTOCOL
# ============================================
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("pROC", quietly = TRUE)) install.packages("pROC")
library(pROC)

cat("✅ Setup complete! pROC package loaded.")

---

# Part 1: The Confusion Matrix

## 40 minutes

---

## 1.1 Clinical Context

Every diagnostic test has to answer: **Is this patient positive or negative?**

But tests aren't perfect. They can be:
- **True Positive (TP):** Correctly identified positive
- **True Negative (TN):** Correctly identified negative
- **False Positive (FP):** Incorrectly called positive (Type I error)
- **False Negative (FN):** Incorrectly called negative (Type II error)

## 1.2 The 2x2 Confusion Matrix

```
                    ACTUAL CONDITION
                    Positive    Negative
TEST        Positive    TP          FP
RESULT      Negative    FN          TN
```

In [ ]:
# Example: Dengue rapid test results
# Tested 100 patients, compared to gold standard PCR

TP <- 45  # Test positive, actually positive
FP <- 5   # Test positive, actually negative
FN <- 8   # Test negative, actually positive
TN <- 42  # Test negative, actually negative

total <- TP + FP + FN + TN
cat("Total patients:", total)

## 1.3 Key Performance Metrics

### Sensitivity (True Positive Rate)

**"How good is the test at detecting POSITIVES?"**

$$\text{Sensitivity} = \frac{TP}{TP + FN}$$

High sensitivity = Good for **ruling OUT** disease (SnNout)

In [ ]:
# Calculate Sensitivity
sensitivity <- TP / (TP + FN)
cat(sprintf("Sensitivity: %.1f%% (%.2f)\n", sensitivity * 100, sensitivity))
cat("Interpretation: Of all ACTUAL positives, the test correctly identified", 
    round(sensitivity * 100), "%")

### Specificity (True Negative Rate)

**"How good is the test at detecting NEGATIVES?"**

$$\text{Specificity} = \frac{TN}{TN + FP}$$

High specificity = Good for **ruling IN** disease (SpPin)

In [ ]:
# Calculate Specificity
specificity <- TN / (TN + FP)
cat(sprintf("Specificity: %.1f%% (%.2f)\n", specificity * 100, specificity))
cat("Interpretation: Of all ACTUAL negatives, the test correctly identified", 
    round(specificity * 100), "%")

### PPV and NPV (Predictive Values)

**PPV (Positive Predictive Value):** If the test says POSITIVE, what's the probability the patient is truly positive?

$$PPV = \frac{TP}{TP + FP}$$

**NPV (Negative Predictive Value):** If the test says NEGATIVE, what's the probability the patient is truly negative?

$$NPV = \frac{TN}{TN + FN}$$

In [ ]:
# Calculate PPV and NPV
PPV <- TP / (TP + FP)
NPV <- TN / (TN + FN)

cat(sprintf("PPV: %.1f%% — If test positive, %.0f%% chance truly positive\n", PPV * 100, PPV * 100))
cat(sprintf("NPV: %.1f%% — If test negative, %.0f%% chance truly negative\n", NPV * 100, NPV * 100))

## 1.4 Function to Calculate All Metrics

In [ ]:
# Function to calculate diagnostic metrics
calc_metrics <- function(TP, FP, FN, TN) {
  sens <- TP / (TP + FN)
  spec <- TN / (TN + FP)
  ppv <- TP / (TP + FP)
  npv <- TN / (TN + FN)
  accuracy <- (TP + TN) / (TP + FP + FN + TN)
  
  cat("=== Diagnostic Performance ===\n")
  cat(sprintf("Sensitivity: %.1f%%\n", sens * 100))
  cat(sprintf("Specificity: %.1f%%\n", spec * 100))
  cat(sprintf("PPV: %.1f%%\n", ppv * 100))
  cat(sprintf("NPV: %.1f%%\n", npv * 100))
  cat(sprintf("Accuracy: %.1f%%\n", accuracy * 100))
}

# Test it
calc_metrics(TP = 45, FP = 5, FN = 8, TN = 42)

## 1.5 The Sensitivity-Specificity Trade-off

**You can't maximize both!**

| Scenario | Priority | Why |
|----------|----------|-----|
| Screening for cancer | High **Sensitivity** | Don't miss any cases |
| Confirmatory test | High **Specificity** | Avoid false positives |
| Blood bank testing | BOTH high | Safety critical |

---

# Part 2: ROC Analysis

## 40 minutes

---

## 2.1 What is ROC?

**ROC = Receiver Operating Characteristic**

A ROC curve shows how sensitivity and specificity change as you move the **cutoff threshold**.

- X-axis: 1 - Specificity (False Positive Rate)
- Y-axis: Sensitivity (True Positive Rate)

In [ ]:
# Example: Dengue ELISA data
# Higher OD = more likely positive

elisa_data <- data.frame(
  SampleID = paste0("S", sprintf("%03d", 1:40)),
  OD = c(0.85, 0.32, 1.45, 0.18, 0.92, 0.55, 1.28, 0.25, 0.78, 0.41,
         1.62, 0.38, 0.95, 0.22, 1.15, 0.48, 0.72, 1.35, 0.29, 0.88,
         0.65, 0.35, 1.52, 0.28, 0.98, 0.42, 0.58, 1.18, 0.31, 0.82,
         0.68, 0.24, 1.42, 0.45, 0.75, 0.52, 1.08, 0.33, 0.62, 0.95),
  PCR = c(1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
          1, 0, 1, 0, 1, 0, 0, 1, 0, 1,
          1, 0, 1, 0, 1, 0, 0, 1, 0, 1,
          1, 0, 1, 0, 1, 0, 1, 0, 0, 1)  # 1 = PCR positive
)

head(elisa_data, 10)

## 2.2 Using the pROC Package

In [ ]:
# Generate ROC curve
roc_result <- roc(elisa_data$PCR, elisa_data$OD)

# Print AUC
cat("AUC (Area Under Curve):", round(auc(roc_result), 3), "\n")
cat("\nInterpretation:\n")
cat("  0.5 = No discrimination (random guess)\n")
cat("  0.7-0.8 = Acceptable\n")
cat("  0.8-0.9 = Excellent\n")
cat("  >0.9 = Outstanding")

In [ ]:
# Plot the ROC curve
plot(roc_result,
     main = "ROC Curve: Dengue ELISA",
     col = "steelblue",
     lwd = 3,
     print.auc = TRUE,
     print.auc.x = 0.4,
     print.auc.y = 0.2)

# Add diagonal reference line (random classifier)
abline(a = 0, b = 1, lty = 2, col = "gray")

## 2.3 Finding the Optimal Cutoff

**Youden Index** = Sensitivity + Specificity - 1

The cutoff that maximizes Youden Index gives the "best balance" of sensitivity and specificity.

In [ ]:
# Find optimal cutoff using Youden Index
optimal <- coords(roc_result, "best", ret = c("threshold", "sensitivity", "specificity"))

cat("=== Optimal Cutoff (Youden Index) ===\n\n")
cat(sprintf("Cutoff OD: %.2f\n", optimal$threshold))
cat(sprintf("Sensitivity: %.1f%%\n", optimal$sensitivity * 100))
cat(sprintf("Specificity: %.1f%%\n", optimal$specificity * 100))

In [ ]:
# Plot with optimal cutoff marked
plot(roc_result, main = "ROC Curve with Optimal Cutoff", col = "steelblue", lwd = 3)
points(optimal$specificity, optimal$sensitivity, pch = 19, col = "red", cex = 2)
text(optimal$specificity - 0.1, optimal$sensitivity + 0.05,
     sprintf("Cutoff = %.2f", optimal$threshold), col = "red")

## 2.4 Exploring Different Cutoffs

Let's see how performance changes at different cutoff values.

In [ ]:
# Get metrics at various cutoffs
cutoffs <- c(0.4, 0.5, 0.6, 0.7, 0.8, 0.64)

cat("Cutoff\tSens\tSpec\tYouden\n")
cat("------\t----\t----\t------\n")

for (cut in cutoffs) {
  metrics <- coords(roc_result, cut, ret = c("sensitivity", "specificity"))
  youden <- metrics$sensitivity + metrics$specificity - 1
  cat(sprintf("%.2f\t%.1f%%\t%.1f%%\t%.2f\n", 
              cut, metrics$sensitivity * 100, metrics$specificity * 100, youden))
}

## 2.5 Clinical Application: Choosing a Cutoff

The "best" cutoff depends on your **clinical goal**.

In [ ]:
# For SCREENING: Threshold for sensitivity (e.g., 90%)
screen_cutoff <- coords(roc_result, x = 0.90, input = "sensitivity", 
                         ret = c("threshold", "sensitivity", "specificity"))

cat("=== Screening Cutoff (Sens ≥ 90%) ===\n")
cat(sprintf("Cutoff: %.2f\n", screen_cutoff$threshold))
cat(sprintf("Sensitivity: %.1f%%\n", screen_cutoff$sensitivity * 100))
cat(sprintf("Specificity: %.1f%%\n", screen_cutoff$specificity * 100))

In [ ]:
# For CONFIRMATION: Prioritize specificity (e.g., 95%)
confirm_cutoff <- coords(roc_result, x = 0.95, input = "specificity", 
                          ret = c("threshold", "sensitivity", "specificity"))

cat("=== Confirmatory Cutoff (Spec ≥ 95%) ===\n")
cat(sprintf("Cutoff: %.2f\n", confirm_cutoff$threshold))
cat(sprintf("Sensitivity: %.1f%%\n", confirm_cutoff$sensitivity * 100))
cat(sprintf("Specificity: %.1f%%\n", confirm_cutoff$specificity * 100))

## 2.6 Comparing Two Tests

In [ ]:
# Simulate a second test (slightly worse)
set.seed(123)
elisa_data$OD_v2 <- elisa_data$OD + rnorm(40, 0, 0.2)

roc1 <- roc(elisa_data$PCR, elisa_data$OD)
roc2 <- roc(elisa_data$PCR, elisa_data$OD_v2)

# Compare AUCs
cat("Test 1 AUC:", round(auc(roc1), 3), "\n")
cat("Test 2 AUC:", round(auc(roc2), 3), "\n")

In [ ]:
# Plot both curves
plot(roc1, col = "steelblue", lwd = 2, main = "Comparing Two ELISA Assays")
plot(roc2, col = "coral", lwd = 2, add = TRUE)
legend("bottomright", 
       legend = c(paste("Test 1: AUC =", round(auc(roc1), 3)),
                  paste("Test 2: AUC =", round(auc(roc2), 3))),
       col = c("steelblue", "coral"), lwd = 2)

---

# Key Takeaways

1. **Confusion matrix** = TP, FP, FN, TN

2. **Sensitivity** = TP/(TP+FN) — good for ruling OUT

3. **Specificity** = TN/(TN+FP) — good for ruling IN

4. **ROC curve** shows trade-off between sensitivity and specificity

5. **AUC** = overall test performance (higher is better)

6. **Optimal cutoff** depends on clinical context

---

## Key Functions

```r
library(pROC)
roc_obj <- roc(response, predictor)
auc(roc_obj)
coords(roc_obj, "best")  # Youden index
plot(roc_obj)
```

---

## Now proceed to Tutorial 7! 🧪